# EDA — NIH ChestX-ray14
**Explanation-Supervised Attention for Multi-Label Thoracic Disease Classification**

This notebook covers:
1. Dataset overview (image count, patient count)
2. Class distribution and imbalance
3. Label co-occurrence matrix
4. Bounding-box coverage analysis
5. Patient-level split verification
6. Sample visualisations

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.splits import (
    load_dataframe, build_balanced_subset, patient_level_split,
    get_class_weights, compute_cooccurrence_matrix, CLASS_NAMES
)
from src.data.masks import load_bbox_lookup
from src.plots import plot_cooccurrence_matrix

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded. Working dir:', os.getcwd())

Config loaded. Working dir: C:\Users\Israe\Documents\Claude\Projects\DL PROJECT WORK


## 1. Dataset overview

In [2]:
df = load_dataframe(cfg['data_dir'])
print(f'Total images   : {len(df):,}')
print(f'Unique patients: {df["patient_id"].nunique():,}')
print(f'Boxed images   : {df["has_box"].sum():,}')
df.head(3)

Total images   : 112,120
Unique patients: 30,805
Boxed images   : 880


,image_id,finding_labels,Follow-up #,patient_id,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,...,Nodule,Pneumonia,Pneumothorax,Consolidation,Edema,Emphysema,Fibrosis,Pleural_Thickening,Hernia,has_box
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,...,0,0,0,0,0,0,0,0,0,0
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,...,0,0,0,0,0,1,0,0,0,0
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,...,0,0,0,0,0,0,0,0,0,0


## 2. Class distribution

In [3]:
counts = df[CLASS_NAMES].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#d62728' if c < 2000 else '#1f77b4' for c in counts.values]
ax.bar(counts.index, counts.values, color=colors)
ax.set_xticklabels(counts.index, rotation=45, ha='right')
ax.set_ylabel('Image count')
ax.set_title('Class Distribution — NIH ChestX-ray14')
ax.grid(axis='y', alpha=0.3)
for i, (cls, val) in enumerate(counts.items()):
    ax.text(i, val + 100, f'{val:,}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('outputs/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(counts)

C:\Users\Israe\AppData\Local\Temp\ipykernel_25792\1652077654.py:6: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(counts.index, rotation=45, ha='right')


Infiltration          19894
Effusion              13317
Atelectasis           11559
Nodule                 6331
Mass                   5782
Pneumothorax           5302
Consolidation          4667
Pleural_Thickening     3385
Cardiomegaly           2776
Emphysema              2516
Edema                  2303
Fibrosis               1686
Pneumonia              1431
Hernia                  227
dtype: uint64


C:\Users\Israe\AppData\Local\Temp\ipykernel_25792\1652077654.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Multi-label statistics

In [4]:
label_counts_per_image = df[CLASS_NAMES].sum(axis=1)
print('Labels per image distribution:')
print(label_counts_per_image.value_counts().sort_index())

fig, ax = plt.subplots(figsize=(8, 4))
label_counts_per_image.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Number of labels per image')
ax.set_ylabel('Count')
ax.set_title('Multi-label distribution')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/multilabel_distribution.png', dpi=150)
plt.show()

Labels per image distribution:
0    60361
1    30963
2    14306
3     4856
4     1247
5      301
6       67
7       16
8        1
9        2
Name: count, dtype: int64


C:\Users\Israe\AppData\Local\Temp\ipykernel_25792\1286014878.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Label co-occurrence matrix

In [5]:
cooc = compute_cooccurrence_matrix(df)
print(f'Co-occurrence matrix shape: {cooc.shape}')

os.makedirs('outputs/figures', exist_ok=True)
plot_cooccurrence_matrix(cooc, CLASS_NAMES, 'outputs/figures')

Co-occurrence matrix shape: (14, 14)
[plots] Co-occurrence matrix → outputs/figures\cooccurrence_matrix.png


## 5. Bounding-box coverage analysis

In [6]:
bbox_csv = os.path.join(cfg['data_dir'], cfg['csv_bbox'])
bbox_df  = pd.read_csv(bbox_csv)
bbox_df.columns = bbox_df.columns.str.strip()

cols = list(bbox_df.columns)
bbox_df = bbox_df.rename(columns={
    cols[0]: 'image_id', cols[1]: 'label',
    cols[2]: 'x', cols[3]: 'y', cols[4]: 'w', cols[5]: 'h'
})[['image_id','label','x','y','w','h']]

print(f'Total bounding boxes : {len(bbox_df):,}')
print(f'Images with boxes    : {bbox_df["image_id"].nunique():,}')
print('\nBoxes per class:')
print(bbox_df['label'].value_counts())

bbox_df['area'] = bbox_df['w'] * bbox_df['h']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(bbox_df['area'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Box area (pixels^2)'); axes[0].set_ylabel('Count')
axes[0].set_title('Bounding-box area distribution')
axes[0].grid(alpha=0.3)

bbox_df['label'].value_counts().plot(kind='barh', ax=axes[1], color='tomato')
axes[1].set_xlabel('Count'); axes[1].set_title('Boxes per pathology class')
axes[1].grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/bbox_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

Total bounding boxes : 984
Images with boxes    : 880

Boxes per class:
label
Atelectasis     180
Effusion        153
Cardiomegaly    146
Infiltrate      123
Pneumonia       120
Pneumothorax     98
Mass             85
Nodule           79
Name: count, dtype: int64


C:\Users\Israe\AppData\Local\Temp\ipykernel_25792\4211146603.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Patient-level split verification

In [7]:
if cfg.get('subset_size'):
    df_use = build_balanced_subset(df, cfg['subset_size'], cfg['random_seed'])
    print(f'Using balanced subset: {len(df_use):,} images')
else:
    df_use = df

train_df, val_df, test_df = patient_level_split(
    df_use, cfg['val_frac'], cfg['test_frac'], cfg['random_seed']
)

tr_pts = set(train_df['patient_id'])
va_pts = set(val_df['patient_id'])
te_pts = set(test_df['patient_id'])
assert tr_pts.isdisjoint(va_pts), 'LEAK: train-val'
assert tr_pts.isdisjoint(te_pts), 'LEAK: train-test'
assert va_pts.isdisjoint(te_pts), 'LEAK: val-test'
print('No patient-level leakage.')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (split_df, name) in zip(axes, [(train_df,'Train'),(val_df,'Val'),(test_df,'Test')]):
    split_df[CLASS_NAMES].sum().sort_values(ascending=False).plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'{name} split ({len(split_df):,} images)')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Count'); ax.grid(axis='y', alpha=0.3)
plt.suptitle('Class distribution per split', y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Using balanced subset: 5,521 images
[splits] ✓ No patient-level leakage detected.
[splits] Patients → train:2,762  val:346  test:346
[splits] Images   → train:4,380  val:556  test:585
[splits] Boxed images → train:682  val:102  test:96
No patient-level leakage.


C:\Users\Israe\AppData\Local\Temp\ipykernel_25792\3679174950.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Class weights (for loss function)

In [8]:
weights = get_class_weights(train_df)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(CLASS_NAMES, weights, color='seagreen')
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_ylabel('Positive weight (neg/pos ratio)')
ax.set_title('Per-class weights for Focal/BCE loss')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/class_weights.png', dpi=150)
plt.show()

for cls, w in zip(CLASS_NAMES, weights):
    print(f'{cls:<22}: {w:.2f}')

C:\Users\Israe\AppData\Local\Temp\ipykernel_25792\3576256322.py:4: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')


Atelectasis           : 3.36
Cardiomegaly          : 7.94
Effusion              : 2.98
Infiltration          : 2.35
Mass                  : 5.83
Nodule                : 6.31
Pneumonia             : 9.71
Pneumothorax          : 6.53
Consolidation         : 7.42
Edema                 : 10.01
Emphysema             : 10.81
Fibrosis              : 12.69
Pleural_Thickening    : 8.71
Hernia                : 24.61


C:\Users\Israe\AppData\Local\Temp\ipykernel_25792\3576256322.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
